# Phase 4 — Evaluate the saved baseline
Attach `awsaf49/cbis-ddsm-breast-cancer-image-dataset` and your saved **Phase 3 checkpoint** (`best.pt`) using Kaggle's Add Input. Enable Internet for the GitHub checkout. GPU speeds up inference; CPU also works.

Run the cells in order. This notebook uses the original committed manifest and the checkpoint's saved split settings. It produces the held-out test report, ROC/PR curves, confusion matrix, and per-sample predictions.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO = Path('/kaggle/working/Explainable_DL_Cancer_Detection')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/siddiq-x/Explainable_DL_Cancer_Detection.git', str(REPO)], check=True)
elif (REPO / '.git').is_dir():
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO, check=True)
assert (REPO / 'evaluate.py').is_file(), f'Project code missing at {REPO}'
os.chdir(REPO)
print('Project:', REPO)


In [ ]:
from importlib.util import find_spec

# Retain Kaggle's existing torch/torchvision installation.
packages = {'numpy': 'numpy', 'pandas': 'pandas', 'PIL': 'Pillow', 'sklearn': 'scikit-learn', 'matplotlib': 'matplotlib', 'yaml': 'PyYAML'}
missing = [package for module, package in packages.items() if find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import torch
import torchvision
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch:', torch.__version__, '| device:', DEVICE)
help_text = subprocess.check_output([sys.executable, str(REPO / 'evaluate.py'), '--help'], text=True)
assert '--manifest-path' in help_text, 'The repo is outdated. Sync the Phase 4 fixes to GitHub, then rerun the checkout cell.'


## Locate the images and checkpoint
Leave the two overrides as `None` for automatic discovery. If several checkpoints are attached, set `CHECKPOINT_OVERRIDE` to the desired `best.pt` path printed below. If none is found, attach the saved training notebook's output or upload the checkpoint as a Kaggle Dataset.

In [ ]:
DATASET_OVERRIDE = None
CHECKPOINT_OVERRIDE = None

inputs = Path('/kaggle/input')
if DATASET_OVERRIDE:
    DATASET = Path(DATASET_OVERRIDE)
else:
    roots = sorted({p.parent.parent for p in inputs.rglob('dicom_info.csv') if p.parent.name == 'csv' and (p.parent.parent / 'jpeg').is_dir()})
    if len(roots) != 1:
        raise RuntimeError(f'Attach the CBIS-DDSM JPEG dataset, or set DATASET_OVERRIDE. Candidate roots: {roots}')
    DATASET = roots[0]
assert (DATASET / 'csv/dicom_info.csv').is_file() and (DATASET / 'jpeg').is_dir()

if CHECKPOINT_OVERRIDE:
    CHECKPOINT = Path(CHECKPOINT_OVERRIDE)
else:
    candidates = list(inputs.rglob('best.pt'))
    for checkpoint_root in (Path('/kaggle/working/checkpoints'), REPO / 'checkpoints'):
        if checkpoint_root.is_dir():
            candidates.extend(checkpoint_root.rglob('best.pt'))
    candidates = sorted(set(candidates))
    print('Checkpoint candidates:', *candidates, sep='\n')
    if len(candidates) != 1:
        raise RuntimeError('Attach your saved Phase 3 best.pt, then set CHECKPOINT_OVERRIDE if more than one candidate is listed.')
    CHECKPOINT = candidates[0]
assert CHECKPOINT.is_file(), f'Checkpoint missing: {CHECKPOINT}'
MANIFEST = REPO / 'data/manifest.csv'
assert MANIFEST.is_file(), 'Original baseline manifest is missing from the checkout.'
snapshot = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
EXPERIMENT = snapshot['config']['experiment_name']
SAVED_CONFIG = snapshot['config']
print('Dataset:', DATASET)
print('Checkpoint:', CHECKPOINT, '| epoch:', snapshot['epoch'] + 1)
print('Seed:', SAVED_CONFIG['seed'], '| image size:', SAVED_CONFIG['data']['image_size'])
del snapshot


In [ ]:
REPORTS = Path('/kaggle/working/phase4_results')
# Explicit paths override any old CANCER_* environment variables.
subprocess.run([
    sys.executable, str(REPO / 'evaluate.py'),
    '--checkpoint', str(CHECKPOINT), '--split', 'test', '--threshold', '0.5',
    '--dataset-dir', str(DATASET), '--manifest-path', str(MANIFEST),
    '--num-workers', '2', '--batch-size', '32', '--device', DEVICE,
    '--output-dir', str(REPORTS),
], cwd=REPO, check=True)


In [ ]:
import json
from IPython.display import display, Image as DisplayImage

result_dir = REPORTS / EXPERIMENT / 'test'
metrics = json.loads((result_dir / 'metrics.json').read_text())
print((result_dir / 'classification_report.txt').read_text())
print('Evaluated rows:', metrics['n_samples'], '(original baseline: 526)')
for name in ('roc_curve.png', 'pr_curve.png', 'confusion_matrix.png'):
    display(DisplayImage(filename=str(result_dir / name)))


## Recover the Phase 3 loss curves
`last.pt` contains history through the final epoch (13 in the recorded baseline run). `best.pt` contains history only through the selected epoch (8). The plot below labels which history is available.

In [ ]:
import matplotlib.pyplot as plt

history_source = CHECKPOINT
last_path = CHECKPOINT.with_name('last.pt')
if last_path.is_file():
    last = torch.load(last_path, map_location='cpu', weights_only=False)
    if last.get('config') == SAVED_CONFIG:
        history_source = last_path
    del last
snapshot = torch.load(history_source, map_location='cpu', weights_only=False)
history = snapshot.get('history', {})
if history.get('train_loss') and history.get('val_loss'):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(epochs, history['train_loss'], label='train')
    ax.plot(epochs, history['val_loss'], label='validation')
    ax.set(xlabel='Epoch', ylabel='Loss', title=f'{EXPERIMENT}: {history_source.name} history')
    ax.legend()
    fig.savefig(REPORTS / EXPERIMENT / 'loss_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
else:
    print('This checkpoint has no saved loss history.')
del snapshot


In [ ]:
import shutil
from IPython.display import FileLink

archive = shutil.make_archive('/kaggle/working/phase4_results', 'zip', root_dir=REPORTS)
print('Saved reports:', archive)
display(FileLink(archive))


## Complete Phase 4
Download `phase4_results.zip` from `/kaggle/working` or save a notebook version that retains these output files. Bring back `metrics.json`, `run_info.json`, and the plots to record the real test results in the project. Phase 4 stays in progress until these results are available.

ROC-AUC and PR-AUC (average precision) use continuous malignant probabilities. The other scores use the fixed 0.5 threshold. Each observation is a manifest abnormality row; patients are kept disjoint across splits. Cross-validation is a separate training run described in `docs/evaluation.md`.